# Filter Addressable Clauses

Review "clause_annotations.csv" and keep clauses that can be checked by reading the SRS.

Approach: Keep clauses about SRS content or requirement quality. Drop clauses that need evidence from other documents or activities (plans, test reports, change logs, sign-offs). If only part of a clause can be checked, explain which part in "review_note".

Output: Save the kept clauses and review notes to "addressable_clauses.csv".

## Load the data

PROCESS: Activities involved in developing and managing software requirements, like reviews, testing, and tracking changes.

SET/SRS: What belongs in the SRS overall and how the requirements fit together, such as required sections and consistent terminology.

MULTIPLE: A mix of categories. One clause may cover individual requirements, the overall SRS, and engineering activities.

In [8]:
#Load the clause annotations CSV, show dimensions and the applies_to categories.
import pandas as pd
clauses = pd.read_csv("../data/clause_annotations.csv")
print(clauses.shape)
clauses["applies_to"].value_counts()

(309, 9)


applies_to
PROCESS     243
MULTIPLE     52
SET/SRS      14
Name: count, dtype: int64

## Remove duplicates

In [9]:
duplicate_columns = ["swe_id", "section", "text"]

#Shows all copies of each duplicate
duplicate_rows = clauses[clauses.duplicated(duplicate_columns, keep=False)]
print(len(duplicate_rows), "rows involved in duplicates:")
display(duplicate_rows[["chunk_id", "swe_id", "section", "source", "type", "applies_to"]])

#Drops the extra copies
rows_before = len(clauses)
clauses = clauses.drop_duplicates(duplicate_columns).reset_index(drop=True)
print("dropped", rows_before - len(clauses), "duplicate rows")

4 rows involved in duplicates:


,chunk_id,swe_id,section,source,type,applies_to
0,npr-1.2-explanatory,NaN,1.2,NPR 7150.2D,explanatory,PROCESS
3,npr-3.6-explanatory,NaN,3.6,NPR 7150.2D,explanatory,PROCESS
14,npr-1.2-explanatory-2,NaN,1.2,NPR 7150.2D,explanatory,PROCESS
118,npr-3.6-explanatory-2,NaN,3.6,NPR 7150.2D,explanatory,PROCESS


dropped 2 duplicate rows


## Drop PROCESS clauses

Filter using the "applies_to" column. Most SRS-checkable content comes from the SWEHB 5.09 guidance rows, not the NPR requirement rows.

PROCESS: They’re about activities like testing, configuration management, peer reviews, and tracking changes or risks.  Nothing in the Starhawk SRS could prove any of those.

In [10]:
#Keep every clause that isn't PROCESS, shows how many are left and their group
addressable = clauses[clauses["applies_to"] != "PROCESS"].copy()
print(len(addressable), "rows left")

# Count the kept clauses by source and label as a sanity check
pd.crosstab(addressable["source"], addressable["applies_to"])

66 rows left


applies_to,MULTIPLE,SET/SRS
source,,
NPR 7150.2D,8,2
SWEHB 5.09,44,12


## Manually review the non-PROCESS rows

Not every row left can be checked using the SRS. All 66 remaining rows were read to see if the SRS alone provides enough information to check them.

Six rows were dropped for one of the following reasons:

Background information: Why SRSs matter, their benefits, and reference lists do not give a specific check.

Tools and organization tips: Tool and spreadsheet tips, with useful checks already covered by kept clauses.

Activities outside the SRS: Coding practices, peer reviews, and static analysis need other records.

In [11]:
# Rows to drop by hand, keyed by chunk_id
manual_drops = {
    #Background information
    "swehb-2-guidance": "Explains why SRSs matter, nothing to pass or fail",
    "swehb-2.1-guidance": "Lists benefits of having an SRS, nothing to pass or fail",
    "swehb-5.1-guidance": "List of reference documents, not a rule",

    #Tools and organization tips
    "swehb-4-guidance": "Tool and spreadsheet tips for small projects; its SRS section list and fields are covered by kept clauses",

    #Activities outside the SRS
    "npr-3.11.1-note": "About coding and testing practices, code doesn't exist at the SRS stage",
    "npr-3.11.5-note": "Peer reviews and static analysis are activities, evidence is outside the SRS",
}

#Check: every entry in manual_drops should match a row
unmatched = [chunk for chunk in manual_drops if chunk not in set(addressable["chunk_id"])]
assert len(unmatched) == 0, "These manual_drops entries matched no row: " + str(unmatched)

#Look up each row's drop reason by chunk_id
addressable["drop_reason"] = addressable["chunk_id"].map(manual_drops)

#Split into dropped rows and kept rows and remove the helper column
dropped_rows = addressable[addressable["drop_reason"].notna()]
addressable = addressable[addressable["drop_reason"].isna()].drop(columns=["drop_reason"])

print("manually dropped", len(dropped_rows), "kept", len(addressable))

#Show what was dropped and why
dropped_rows[["chunk_id", "source", "section", "applies_to", "drop_reason"]]

manually dropped 6 kept 60


,chunk_id,source,section,applies_to,drop_reason
144,npr-3.11.1-note,NPR 7150.2D,3.11.1,MULTIPLE,"About coding and testing practices, code doesn..."
150,npr-3.11.5-note,NPR 7150.2D,3.11.5,MULTIPLE,Peer reviews and static analysis are activitie...
251,swehb-2-guidance,SWEHB 5.09,2,SET/SRS,"Explains why SRSs matter, nothing to pass or fail"
252,swehb-2.1-guidance,SWEHB 5.09,2.1,SET/SRS,"Lists benefits of having an SRS, nothing to pa..."
297,swehb-4-guidance,SWEHB 5.09,4,MULTIPLE,Tool and spreadsheet tips for small projects; ...
298,swehb-5.1-guidance,SWEHB 5.09,5.1,SET/SRS,"List of reference documents, not a rule"


## Review notes on kept rows

Some kept clauses need more attention when checked against the SRS. Adding a "review_note" here to 7 of them, for next person to take a look at. No rows are dropped.

Partial checks: Only part of the clause can be checked using the SRS.

Overlap: Two clauses cover the same safety behaviors.

Mixed content: A section includes both SRS-checkable content and other guidance.

One relevant item: Only one item of the architecture clause can be checked using the SRS.

In [12]:
addressable["review_note"] = ""


def note(chunk_id, text):
    assert chunk_id in set(addressable["chunk_id"]), "No kept row with chunk_id " + chunk_id
    addressable.loc[addressable["chunk_id"] == chunk_id, "review_note"] = text

#Partly checkable
note("npr-4.1.6-requirement", "SRS shows conflicting requirements. Tracking the fixes isn't in the SRS.")
note("npr-4.1.4-requirement", "SRS shows safety constraints. The safety analysis behind them isn't in the SRS.")

#Overlapping
note("npr-3.7.3-requirement", "Same safety list as SWEHB 3.2.1.4. Count as one finding.")
note("swehb-3.2.1.4-guidance", "Has the SWE-134 safety list (a-l). SRS shows they're required, not built.")

#Long mixed sections
note("swehb-6.1-guidance", "Mostly lessons learned. Relevant parts: hardware behavior and units.")
note("swehb-7.1-guidance", "Mostly assurance advice. Relevant parts: vague wording, off-nominal modes, safety, security, timing.")

#One item applies
note("npr-4.2.2-explanatory", "Items a-f are architecture. Item g is about the SRS: valid and invalid operating modes.")

#Show clause and review note
addressable[addressable["review_note"] != ""][["chunk_id", "review_note"]]

,chunk_id,review_note
127,npr-3.7.3-requirement,Same safety list as SWEHB 3.2.1.4. Count as on...
162,npr-4.1.4-requirement,SRS shows safety constraints. The safety analy...
164,npr-4.1.6-requirement,SRS shows conflicting requirements. Tracking t...
167,npr-4.2.2-explanatory,Items a-f are architecture. Item g is about th...
267,swehb-3.2.1.4-guidance,Has the SWE-134 safety list (a-l). SRS shows t...
304,swehb-6.1-guidance,Mostly lessons learned. Relevant parts: hardwa...
306,swehb-7.1-guidance,Mostly assurance advice. Relevant parts: vague...


## SWE-052: Requirement traceability

Traceability links each software requirement to the higher-level need it supports.

SWE-052 was removed because it was labelled PROCESS. It is added back here because these links can be checked in the SRS. Starhawk Section 10 lists mission needs, but says the table linking requirements to those needs has not been developed.

SWE-052 is now labelled MULTIPLE because the SRS can show the links, but cannot prove they are kept updated over time.

If you disagree and want to leave it out: Set `INCLUDE_SWE052 = False` to undo.

In [ ]:
INCLUDE_SWE052 = True

if INCLUDE_SWE052:
    #Find the SWE-052 rows
    swe052_rows = clauses[clauses["swe_id"] == "SWE-052"].copy()

    #Change the label and explain why they are kept
    swe052_rows["applies_to"] = "MULTIPLE"
    swe052_rows["review_note"] = ("Moved in from PROCESS: an SRS can show whether requirements trace to "
                                  "higher-level needs. Example: Starhawk Sec. 10 lists mission needs but "
                                  "says the trace matrix isn't done.")

     #Add SWE-052 to the kept rows and reset row numbers
    addressable = pd.concat([addressable, swe052_rows]).reset_index(drop=True)

print(len(addressable), "final rows")

62 final rows


## Save to "addressable_clauses.csv"

In [14]:
output_columns = ["chunk_id", "swe_id", "section", "text", "source", "source_url", "type",
                  "applies_to", "interpretation", "review_note"]

addressable[output_columns].to_csv("../data/addressable_clauses.csv", index=False)

print("Rows by label:")
print(addressable["applies_to"].value_counts())
print()
print("Rows by source:")
print(addressable["source"].value_counts())

Rows by label:
applies_to
MULTIPLE    51
SET/SRS     11
Name: count, dtype: int64

Rows by source:
source
SWEHB 5.09     52
NPR 7150.2D    10
Name: count, dtype: int64
